In [ ]:
!pip install -q openai-agents python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.6/403.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 6.7 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

Enter your OpenAI API Key: ··········


In [ ]:
from agents import Agent, Runner, tool, function_tool
from agents.handoffs import Handoff
from agents.tracing import trace

In [ ]:
from agents import Agent

math_agent = Agent(
    name="Math Agent",
    instructions="""
    You are a mathematical problem solver.
    Solve problems step by step and give final answer clearly.
    """,
    model="gpt-4.1-mini"
)

finance_agent = Agent(
    name="Finance Agent",
    instructions="""
    You are a financial analyst.
    Provide structured financial reasoning.
    """,
    model="gpt-4.1-mini"
)

In [ ]:
!pip install -q openai-agents

In [ ]:
from agents import Agent, Runner
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

Enter OpenAI API Key: ··········


In [ ]:
math_agent = Agent(
    name="Math Agent",
    instructions="""
    You are a mathematical reasoning expert.
    Solve problems step by step and give final answer clearly.
    """,
    model="gpt-4.1-mini"
)

finance_agent = Agent(
    name="Finance Agent",
    instructions="""
    You are a financial advisor.
    Provide structured reasoning and safe financial advice.
    """,
    model="gpt-4.1-mini"
)

In [ ]:
orchestrator = Agent(
    name="Orchestrator",
    instructions="""
    If the query is about math → handoff to Math Agent.
    If about finance → handoff to Finance Agent.
    Otherwise answer yourself.
    """,
    handoffs=[math_agent, finance_agent],   # ← FIXED
    model="gpt-4.1"
)

In [ ]:
import asyncio
from agents import Agent, Runner

math_agent = Agent(
    name="Math Agent",
    instructions="Solve math problems step by step.",
    model="gpt-4.1-mini"
)

finance_agent = Agent(
    name="Finance Agent",
    instructions="Answer finance questions responsibly.",
    model="gpt-4.1-mini"
)

orchestrator = Agent(
    name="Orchestrator",
    instructions="""
    Route math queries to Math Agent.
    Route finance queries to Finance Agent.
    """,
    handoffs=[math_agent, finance_agent],
    model="gpt-4.1"
)

async def main():
    result = await Runner.run(
        orchestrator,
        "What is 56 * 89?"
    )
    print(result.final_output)

await main()

Let's calculate 56 * 89 step by step.

1. Multiply 56 by 80 (since 89 = 80 + 9):
   56 * 80 = 4480
2. Multiply 56 by 9:
   56 * 9 = 504
3. Add the two results:
   4480 + 504 = 4984

So, 56 * 89 = 4984.


### **Static Handoff (Predefined Delegation)**

In [ ]:
math_agent = Agent(
    name="Math Specialist",
    instructions="Solve mathematical problems step by step.",
    model="gpt-4.1-mini"
)

legal_agent = Agent(
    name="Legal Specialist",
    instructions="Provide structured legal reasoning.",
    model="gpt-4.1-mini"
)

orchestrator = Agent(
    name="Orchestrator",
    instructions="""
    If query involves calculations → use Math Specialist.
    If legal matter → use Legal Specialist.
    Otherwise answer directly.
    """,
    handoffs=[math_agent, legal_agent],
    model="gpt-4.1"
)

async def run_static():
    result = await Runner.run(
        orchestrator,
        "What is 45 * 78?"
    )
    print(result.final_output)

await run_static()

Let's calculate \(45 \times 78\) step by step:

1. Multiply 45 by 70 (since 78 = 70 + 8):
   \(45 \times 70 = 3150\)

2. Multiply 45 by 8:
   \(45 \times 8 = 360\)

3. Add the two results:
   \(3150 + 360 = 3510\)

So, \(45 \times 78 = 3510\).


### **Conditional Programmatic Handoff**

In [ ]:
async def conditional_router(user_query: str):
    if "calculate" in user_query.lower():
        return await Runner.run(math_agent, user_query)
    elif "law" in user_query.lower():
        return await Runner.run(legal_agent, user_query)
    else:
        return await Runner.run(orchestrator, user_query)

result = await conditional_router("Calculate 100 * 45")
print(result.final_output)

To calculate \( 100 \times 45 \):

Step 1: Multiply the numbers directly.
\[ 100 \times 45 = 4500 \]

So, the result is \( \boxed{4500} \).


### **Tool-Based Handoff (Agent-as-Tool)**

In [ ]:
@function_tool
async def math_tool(query: str) -> str:
    result = await Runner.run(math_agent, query)
    return result.final_output

main_agent = Agent(
    name="Main Agent",
    instructions="""
    If math-related question appears, call math_tool.
    """,
    tools=[math_tool],
    model="gpt-4.1"
)

result = await Runner.run(main_agent, "What is 234 * 567?")
print(result.final_output)

234 × 567 = 132,678.


### **TYPE 4 — Multi-Hop Sequential Handoff**

In [ ]:
analysis_agent = Agent(
    name="Analysis Agent",
    instructions="Analyze problem deeply.",
    model="gpt-4.1-mini"
)

execution_agent = Agent(
    name="Execution Agent",
    instructions="Perform required calculation or solution.",
    model="gpt-4.1-mini"
)

analysis_agent.handoffs = [execution_agent]

async def multi_hop():
    result = await Runner.run(
        analysis_agent,
        "Break down and solve 345 * 987"
    )
    print(result.final_output)

await multi_hop()

Let's break down the multiplication of 345 by 987 step by step using the traditional multiplication method:

Step 1: Multiply 345 by 7 (the units digit of 987)
345
×  7
____
2415

Step 2: Multiply 345 by 8 (the tens digit of 987), then multiply the result by 10
345
×  8
____
2760
Now multiply by 10 (add one zero at the end): 27600

Step 3: Multiply 345 by 9 (the hundreds digit of 987), then multiply the result by 100
345
×  9
____
3105
Now multiply by 100 (add two zeros at the end): 310500

Step 4: Add all partial products together
    2415
   27600
 +310500
________
 340515

So, 345 * 987 = 340,515.


### **TYPE 5 — Router Agent (Dynamic LLM Routing)**

In [ ]:
router = Agent(
    name="Smart Router",
    instructions="""
    Carefully analyze intent.
    Route to appropriate agent.
    """,
    handoffs=[math_agent, legal_agent],
    model="gpt-4.1"
)

result = await Runner.run(router, "Explain contract breach law.")
print(result.final_output)

Certainly! Here’s a clear overview of **contract breach law**:

---

### What is a Contract Breach?

**A contract breach** occurs when one party to a binding agreement fails to fulfill its obligations as agreed upon in the contract, without a valid legal excuse.

---

### Types of Contract Breaches

1. **Minor (Partial) Breach:**
   - The main contract is fulfilled, but a small part isn’t carried out as specified.
   - Example: Delivering goods a day late, but the delay doesn’t cause major damages.

2. **Material (Major) Breach:**
   - A significant failure that defeats the purpose of the contract, allowing the non-breaching party to suspend their own performance and/or sue for damages.
   - Example: Delivering the wrong product entirely.

3. **Anticipatory Breach:**
   - One party announces, or it is clear, that they won’t fulfill their contractual obligations in the future.
   - Example: The supplier notifies they won’t deliver the ordered goods next month.

4. **Actual Breach:**
   

### **TYPE 6 — Parallel Multi-Agent Execution**

In [ ]:
async def parallel_agents():
    task1 = Runner.run(math_agent, "Solve 123 * 456")
    task2 = Runner.run(legal_agent, "Define tort law")

    results = await asyncio.gather(task1, task2)

    for r in results:
        print("\n--- Output ---")
        print(r.final_output)

await parallel_agents()


--- Output ---
Let's solve \(123 \times 456\) step by step:

1. Multiply 123 by 6:  
\(123 \times 6 = 738\)

2. Multiply 123 by 50 (because 456 has 5 in the tens place, which is 50):  
\(123 \times 50 = 6150\)

3. Multiply 123 by 400 (because 456 has 4 in the hundreds place, which is 400):  
\(123 \times 400 = 49200\)

4. Now add all these partial products:  
\(738 + 6150 + 49200 = 56088\)

So,  
\[
123 \times 456 = 56088
\]

**Final answer: 56088**

--- Output ---
**Tort Law: Definition and Structured Explanation**

1. **Definition:**
   Tort law is a branch of civil law that deals with the legal responsibilities and remedies arising from civil wrongs or injuries caused by one party to another. It aims to provide relief to individuals harmed by the wrongful acts or omissions of others, outside of contractual obligations.

2. **Purpose:**
   - To compensate victims for losses or injuries suffered.
   - To deter wrongful conduct.
   - To promote socially acceptable behavior by holding 

In [ ]:
import asyncio
from typing import List

from pydantic import BaseModel

from agents import (
    Agent,
    GuardrailFunctionOutput,
    GuardrailTripwireTriggered,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    input_guardrail,
    Guardrail,
)


# -----------------------------
# 1️⃣ Output Schema
# -----------------------------
class ChurnDetectionOutput(BaseModel):
    is_churn_risk: bool
    reasoning: str


# -----------------------------
# 2️⃣ Churn Detection Agent
# -----------------------------
churn_detection_agent = Agent(
    name="Churn Detection Agent",
    instructions=(
        "Identify if the user message indicates a potential "
        "customer churn risk. "
        "Return is_churn_risk as True or False and explain reasoning."
    ),
    output_type=ChurnDetectionOutput,
)


# -----------------------------
# 3️⃣ Guardrail Function
# -----------------------------
@input_guardrail
async def churn_detection_tripwire(
    ctx: RunContextWrapper,
    agent: Agent,
    input: List[TResponseInputItem],
) -> GuardrailFunctionOutput:

    # Run churn detection agent on the input
    result = await Runner.run(
        churn_detection_agent,
        input,
        context=ctx.context,
    )

    # Return guardrail decision
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_churn_risk,
    )


# -----------------------------
# 4️⃣ Customer Support Agent
# -----------------------------
customer_support_agent = Agent(
    name="Customer Support Agent",
    instructions=(
        "You are a customer support agent. "
        "You help customers with their questions."
    ),
    input_guardrails=[
        Guardrail(guardrail_function=churn_detection_tripwire),
    ],
)


# -----------------------------
# 5️⃣ Main Execution
# -----------------------------
async def main():

    print("\n---- SAFE MESSAGE ----")
    try:
        result = await Runner.run(
            customer_support_agent,
            "Hello! I have a question about my billing."
        )
        print("Response:", result.final_output)
    except GuardrailTripwireTriggered:
        print("Guardrail triggered on safe message (unexpected).")

    print("\n---- CHURN MESSAGE ----")
    try:
        await Runner.run(
            customer_support_agent,
            "I am very unhappy with your service and I want to cancel my subscription."
        )
    except GuardrailTripwireTriggered as e:
        print("⚠ Guardrail Triggered!")
        print("Churn Reason:", e.output_info.reasoning)


if __name__ == "__main__":
    asyncio.run(main())

ImportError: cannot import name 'GuardrailTripwireTriggered' from 'agents' (/usr/local/lib/python3.12/dist-packages/agents/__init__.py)